# Test the Pipeline Package locally with GPU

This notebook introduces how to execute a created Pipeline in your local Python environment.

During its execution, an `ImageSet` payload is created from images, and uses the Pipeline we created in notebook [31-CreatePipelinePackageWithGPU](./31-CreatePipelinePackageWithGPU.ipynb).

To run the GPU variant on target hardware, you need an AI Inference Server version with GPU support (2.9.0 or newer).

The main goal is to understand 
- how the test inputs are formed,
- how the PythonComponent test environment is created,
- how to check the Pipeline behaviour.

#### Create ImageSet input from a JPEG or PNG file

In this case we are using simaticai class `VCAStream` introduced in AI SDK 2.4.0.

This class transforms your images into vision payloads as they arrive from Vision Connector Application (VCA) and provided for your Pipeline Step by the AI Inference Server.

To create the VCAStream for testing, we need to define the folder where the image files sit as `data`, and the variable name of the Pipeline (component) input is defined as `variable_name`. The parameter `filter` is optional, and used for filtering the images you are interested in the folder.

In [ ]:
from simaticai.testing.vca_stream import VCAStream

vca_stream = VCAStream(data="../images/", variable_name="vision_payload", image_format="BGR8", filter="*.jpg")

#### Test the Pipeline Package locally

Take the created `vca_stream` as the test input for the `LocalPipelineRunner`, and run the Pipeline with the test data. The LocalPipelineRunner module is designed to be used __only__ inside a `with` block.

The `run_pipeline` method can be called multiple times but be aware that the internal state of the components is reset between the calls.

In [ ]:
from simaticai.testing.pipeline_runner import LocalPipelineRunner
from pathlib import Path

image_on_edge_package = Path('../packages/Segmentation-on-GPU-edge_1.zip')
test_dir = Path("..").resolve() / "test"

pipeline_output = []
with LocalPipelineRunner(image_on_edge_package, test_dir) as runner:
    runner.update_parameters({"__AI_IS_IMAGE_SET_VISUALIZATION": True})  # Enable visualization
    pipeline_output = runner.run_pipeline(vca_stream)

pipeline_output

We can extract the classes and their areas from the output payload.

In [ ]:
import json

json.loads(pipeline_output[0]['areas'])

We can see the annotated image result, assuming that visualization was set to true.

In [ ]:
from matplotlib import pyplot as plt
from simaticai.payloads.imageset import ImageSet

if "result_image_set" in pipeline_output[0]:
    image_set = ImageSet.from_dict(pipeline_output[0]["result_image_set"])
    # image_detail = pipeline_output[0]["result_image_set"]["detail"][0]
    image_detail = image_set.detail[0]
    # result_img = image_detail["image"]
    result_img = image_set.get_image_rgb(0)
    height = image_detail.height
    width = image_detail.width

    # image_data = numpy.frombuffer(result_img, dtype=numpy.uint8)
    # image_data = image_data.reshape(height, width, 3)

    plt.axis("off")
    plt.imshow(result_img)

Once the expected outputs can be generated here, you are ready to step one further and give the Pipeline a try on AI Inference Server.
The main differences on the real device are
- underlying CPU, GPU configuration,
- underlying Operating System,
- used Python packages.